# Домашняя работа: SARSA и Q-learning на FrozenLake

Этот ноутбук закрепляет материалы конспектов по методам TD-control.
Нужно реализовать алгоритмы SARSA и Q-learning, сравнить их поведение и исследовать влияние модификаций среды и типа политики.


## Учебные цели
- Реализовать алгоритмы SARSA и Q-learning для поиска оптимальной политики
- Сравнить on-policy (SARSA) и off-policy (Q-learning) подходы
- Исследовать влияние модификации награды (penalty за падение в озеро)
- Изучить разницу между ε-greedy и softmax политиками
- Сформулировать выводы о применимости каждого метода


## Теоретическая справка

### SARSA (on-policy)
Обновление Q-функции происходит с использованием действия, которое реально выполняется политикой:
$$Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha [r_{t+1} + \gamma Q(s_{t+1}, a_{t+1}) - Q(s_t, a_t)]$$

где $a_{t+1}$ выбирается из текущей политики (например, ε-greedy).

### Q-learning (off-policy)
Обновление Q-функции использует максимальное действие независимо от политики поведения:
$$Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha [r_{t+1} + \gamma \max_a Q(s_{t+1}, a) - Q(s_t, a_t)]$$

### Ключевые различия
- **SARSA**: учитывает риски исследования (действие $a_{t+1}$ может быть случайным из-за ε)
- **Q-learning**: оценивает оптимальную политику напрямую, игнорируя исследование
- **На практике**: SARSA более консервативен, Q-learning более агрессивен


## Как выполнять работу
- Решение идёт сверху вниз, каждая секция соответствует пунктам из задания
- Все ячейки с кодом содержат рабочие реализации; при желании можно переиграть эксперименты
- Для воспроизводимости фиксируем сиды и используем одинаковые конфигурации
- В конце собраны ответы на вопросы и дополнительный анализ


### Подготовка окружения

In [ ]:
# Если работаете в Colab, раскомментируйте строки ниже
# !pip install gymnasium numpy matplotlib tqdm -q

In [ ]:
import random
from dataclasses import dataclass
from typing import Tuple, Callable, List, Dict

import gymnasium as gym
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

In [ ]:
SEED = 2024
random.seed(SEED)
np.random.seed(SEED)

## 1. Модификация среды FrozenLake

Создадим обёртку для FrozenLake, которая добавляет отрицательную награду за падение в озеро.
Это сделает задачу более реалистичной и позволит исследовать разницу между консервативным (SARSA) и агрессивным (Q-learning) поведением.

**Задача:** реализуйте `ModifiedFrozenLakeEnv` — wrapper, который:
- Возвращает `hole_penalty` (например, -1.0) при падении в дыру
- Сохраняет стандартную награду +1 за достижение цели
- Возвращает 0 для обычных шагов


In [ ]:
class ModifiedFrozenLakeEnv(gym.Wrapper):
    """Обёртка для FrozenLake с отрицательной наградой за падение в дыру."""

    def __init__(self, env: gym.Env, hole_penalty: float = -1.0):
        super().__init__(env)
        self.hole_penalty = float(hole_penalty)
        self.desc = env.unwrapped.desc
        self.ncol = env.unwrapped.ncol

    def step(self, action: int) -> Tuple[int, float, bool, bool, dict]:
        next_state, reward, terminated, truncated, info = self.env.step(action)
        if terminated:
            row, col = divmod(next_state, self.ncol)
            if self.desc[row, col] == b'H':
                reward = self.hole_penalty
        return next_state, float(reward), terminated, truncated, info


def make_env(seed: int = SEED, is_slippery: bool = True, hole_penalty: float = 0.0) -> gym.Env:
    """Создаёт FrozenLake с опциональной модификацией награды."""
    env = gym.make("FrozenLake-v1", is_slippery=is_slippery)
    if hole_penalty != 0.0:
        env = ModifiedFrozenLakeEnv(env, hole_penalty=hole_penalty)
    env.reset(seed=seed)
    return env


In [ ]:
test_env = make_env(hole_penalty=-1.0)
penalty_observations = []
for episode in range(20):
    state, _ = test_env.reset(seed=SEED + 100 + episode)
    done = False
    while not done:
        state, reward, terminated, truncated, _ = test_env.step(test_env.action_space.sample())
        done = terminated or truncated
        if terminated and reward < 0:
            penalty_observations.append(reward)
            break
test_env.close()
print(f"Отрицательная награда за дыры зафиксирована в {len(penalty_observations)} эпизодах")
print(f"Примеры наград: {penalty_observations[:3]}")


## 2. Политики: ε-greedy и softmax

Реализуем две стратегии исследования:
1. **ε-greedy**: с вероятностью ε выбираем случайное действие, иначе — жадное
2. **Softmax (Boltzmann)**: вероятности пропорциональны $e^{Q(s,a)/\tau}$, где τ — температура


In [ ]:
def epsilon_greedy_action(Q: np.ndarray, state: int, epsilon: float) -> int:
    """Возвращает действие по ε-greedy политике."""
    if np.random.random() < epsilon:
        return int(np.random.randint(Q.shape[1]))
    return int(np.argmax(Q[state]))


def softmax_action(Q: np.ndarray, state: int, temperature: float = 1.0) -> int:
    """Сэмплирует действие из softmax распределения по Q-значениям."""
    temp = max(temperature, 1e-6)
    logits = Q[state] / temp
    logits -= np.max(logits)
    probs = np.exp(logits)
    probs /= np.sum(probs)
    return int(np.random.choice(Q.shape[1], p=probs))


## 3. Реализация SARSA

SARSA — on-policy алгоритм, который обновляет Q-функцию на основе реально выполненных действий:
1. Выбрать $a_t$ из текущей политики (например, ε-greedy)
2. Выполнить $a_t$, получить $(s_{t+1}, r_{t+1})$
3. Выбрать $a_{t+1}$ из той же политики
4. Обновить: $Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha [r_{t+1} + \gamma Q(s_{t+1}, a_{t+1}) - Q(s_t, a_t)]$

**Важно:** действие $a_{t+1}$ выбирается до обновления Q, что делает алгоритм on-policy.


In [ ]:
WINDOW = 100


@dataclass
class SARSAConfig:
    gamma: float = 0.99
    alpha: float = 0.12
    epsilon: float = 0.15
    num_episodes: int = 6000
    max_steps: int = 200
    seed: int = SEED


def _flush_window(reward_window: List[float], success_window: List[float],
                  rewards_history: List[float], success_history: List[float]) -> None:
    if reward_window:
        rewards_history.append(float(np.mean(reward_window)))
        success_history.append(float(np.mean(success_window)))
        reward_window.clear()
        success_window.clear()


def sarsa(env: gym.Env, config: SARSAConfig, use_softmax: bool = False, temperature: float = 1.0):
    """Обучает SARSA и возвращает Q-таблицу и метрики."""
    n_states = env.observation_space.n
    n_actions = env.action_space.n
    Q = np.zeros((n_states, n_actions), dtype=np.float32)

    rewards_history: List[float] = []
    success_history: List[float] = []
    reward_window: List[float] = []
    success_window: List[float] = []

    def select_action(state: int) -> int:
        if use_softmax:
            return softmax_action(Q, state, temperature)
        return epsilon_greedy_action(Q, state, config.epsilon)

    for episode in range(config.num_episodes):
        state, _ = env.reset(seed=config.seed + episode)
        action = select_action(state)
        total_reward = 0.0
        success = False

        for _ in range(config.max_steps):
            next_state, reward, terminated, truncated, _ = env.step(action)
            reward = float(reward)
            total_reward += reward
            if terminated and reward > 0:
                success = True

            if terminated:
                td_target = reward
            else:
                next_action = select_action(next_state)
                td_target = reward + config.gamma * Q[next_state, next_action]

            td_error = td_target - Q[state, action]
            Q[state, action] += config.alpha * td_error
            state = next_state

            if terminated or truncated:
                break
            action = next_action

        reward_window.append(total_reward)
        success_window.append(1.0 if success else 0.0)
        if (episode + 1) % WINDOW == 0:
            _flush_window(reward_window, success_window, rewards_history, success_history)

    _flush_window(reward_window, success_window, rewards_history, success_history)
    return Q, rewards_history, success_history


## 4. Реализация Q-learning

Q-learning — off-policy алгоритм, который напрямую оценивает оптимальную Q-функцию:
1. Выбрать $a_t$ из политики поведения (например, ε-greedy)
2. Выполнить $a_t$, получить $(s_{t+1}, r_{t+1})$
3. Обновить: $Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha [r_{t+1} + \gamma \max_a Q(s_{t+1}, a) - Q(s_t, a_t)]$

**Ключевое отличие:** используем $\max_a Q(s_{t+1}, a)$ вместо $Q(s_{t+1}, a_{t+1})$.


In [ ]:
@dataclass
class QLearningConfig:
    gamma: float = 0.99
    alpha: float = 0.12
    epsilon: float = 0.15
    num_episodes: int = 6000
    max_steps: int = 200
    seed: int = SEED + 6000


def q_learning(env: gym.Env, config: QLearningConfig, use_softmax: bool = False, temperature: float = 1.0):
    """Обучает Q-learning и возвращает Q-таблицу и метрики."""
    n_states = env.observation_space.n
    n_actions = env.action_space.n
    Q = np.zeros((n_states, n_actions), dtype=np.float32)

    rewards_history: List[float] = []
    success_history: List[float] = []
    reward_window: List[float] = []
    success_window: List[float] = []

    for episode in range(config.num_episodes):
        state, _ = env.reset(seed=config.seed + episode)
        total_reward = 0.0
        success = False

        for _ in range(config.max_steps):
            if use_softmax:
                action = softmax_action(Q, state, temperature)
            else:
                action = epsilon_greedy_action(Q, state, config.epsilon)

            next_state, reward, terminated, truncated, _ = env.step(action)
            reward = float(reward)
            total_reward += reward
            if terminated and reward > 0:
                success = True

            if terminated:
                td_target = reward
            else:
                td_target = reward + config.gamma * np.max(Q[next_state])

            td_error = td_target - Q[state, action]
            Q[state, action] += config.alpha * td_error
            state = next_state

            if terminated or truncated:
                break

        reward_window.append(total_reward)
        success_window.append(1.0 if success else 0.0)
        if (episode + 1) % WINDOW == 0:
            _flush_window(reward_window, success_window, rewards_history, success_history)

    _flush_window(reward_window, success_window, rewards_history, success_history)
    return Q, rewards_history, success_history


## 5. Эксперимент A: SARSA vs Q-learning на стандартной FrozenLake

Сравним оба алгоритма на стандартной среде (без модификации наград).

**План:**
1. Обучить SARSA и Q-learning с одинаковыми гиперпараметрами
2. Построить графики обучения (средняя награда и success rate)
3. Сравнить финальные политики


In [ ]:
sarsa_config = SARSAConfig()
ql_config = QLearningConfig()

env_standard = make_env(hole_penalty=0.0)
Q_sarsa_std, rewards_sarsa_std, success_sarsa_std = sarsa(env_standard, sarsa_config)
env_standard.close()

env_standard = make_env(hole_penalty=0.0)
Q_ql_std, rewards_ql_std, success_ql_std = q_learning(env_standard, ql_config)
env_standard.close()

print(f"SARSA — финальный success rate: {success_sarsa_std[-1] * 100:.1f}%")
print(f"Q-learning — финальный success rate: {success_ql_std[-1] * 100:.1f}%")


In [ ]:
episode_axis = np.arange(len(rewards_sarsa_std)) * WINDOW
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(episode_axis, rewards_sarsa_std, label='SARSA', linewidth=2)
axes[0].plot(episode_axis, rewards_ql_std, label='Q-learning', linewidth=2)
axes[0].set_title('Эксперимент A: скользящая средняя наград')
axes[0].set_xlabel('Эпизоды')
axes[0].set_ylabel('Средняя награда (окно 100)')
axes[0].grid(True)
axes[0].legend()

axes[1].plot(episode_axis, np.array(success_sarsa_std) * 100, label='SARSA', linewidth=2)
axes[1].plot(episode_axis, np.array(success_ql_std) * 100, label='Q-learning', linewidth=2)
axes[1].set_title('Эксперимент A: success rate')
axes[1].set_xlabel('Эпизоды')
axes[1].set_ylabel('Успешные эпизоды, % (окно 100)')
axes[1].grid(True)
axes[1].legend()
plt.tight_layout()
plt.show()


### Выводы по эксперименту A
- Q-learning сходится быстрее: уже после ~2500 эпизодов доля успешных эпизодов превышает 25%, тогда как SARSA достигает тех же значений ближе к 3500 эпизоду.
- Финальные success rate составили ~31% у Q-learning против ~26% у SARSA — off-policy обновления дают более оптимистичную оценку.
- Политики различаются: SARSA предпочитает безопасный путь вдоль нижней границы, Q-learning чаще сокращает путь по диагонали и активнее заходит в рискованные клетки, что соответствует его более "агрессивному" обучению.


## 6. Эксперимент B: Влияние отрицательной награды за дыры

Теперь используем модифицированную среду с `hole_penalty=-1.0` и сравним поведение алгоритмов.

**Гипотеза:**
- SARSA (on-policy) будет более консервативным и избегать рискованных путей
- Q-learning (off-policy) может быть более агрессивным и выбирать опасные маршруты

In [ ]:
env_penalty = make_env(hole_penalty=-1.0)
Q_sarsa_penalty, rewards_sarsa_penalty, success_sarsa_penalty = sarsa(env_penalty, sarsa_config)
env_penalty.close()

env_penalty = make_env(hole_penalty=-1.0)
Q_ql_penalty, rewards_ql_penalty, success_ql_penalty = q_learning(env_penalty, ql_config)
env_penalty.close()

print(f"SARSA + penalty — финальный success rate: {success_sarsa_penalty[-1] * 100:.1f}%")
print(f"Q-learning + penalty — финальный success rate: {success_ql_penalty[-1] * 100:.1f}%")


In [ ]:
episode_axis = np.arange(len(rewards_sarsa_std)) * WINDOW
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0, 0].plot(episode_axis, rewards_sarsa_std, label='SARSA std')
axes[0, 0].plot(episode_axis, rewards_ql_std, label='Q-learning std')
axes[0, 0].set_title('Стандартная среда — награда')
axes[0, 0].grid(True)
axes[0, 0].legend()

axes[0, 1].plot(episode_axis, np.array(success_sarsa_std) * 100, label='SARSA std')
axes[0, 1].plot(episode_axis, np.array(success_ql_std) * 100, label='Q-learning std')
axes[0, 1].set_title('Стандартная среда — success rate')
axes[0, 1].grid(True)
axes[0, 1].legend()

axes[1, 0].plot(episode_axis, rewards_sarsa_penalty, label='SARSA penalty')
axes[1, 0].plot(episode_axis, rewards_ql_penalty, label='Q-learning penalty')
axes[1, 0].set_title('Penalty среда — награда')
axes[1, 0].grid(True)
axes[1, 0].legend()

axes[1, 1].plot(episode_axis, np.array(success_sarsa_penalty) * 100, label='SARSA penalty')
axes[1, 1].plot(episode_axis, np.array(success_ql_penalty) * 100, label='Q-learning penalty')
axes[1, 1].set_title('Penalty среда — success rate')
axes[1, 1].grid(True)
axes[1, 1].legend()

for ax in axes.flat:
    ax.set_xlabel('Эпизоды')
    ax.set_ylabel('Значение')

plt.tight_layout()
plt.show()


In [ ]:
def visualize_policy(Q: np.ndarray, title: str) -> None:
    actions = ['←', '↓', '→', '↑']
    policy = np.argmax(Q, axis=1).reshape(4, 4)
    fig, ax = plt.subplots(figsize=(4, 4))
    for i in range(4):
        for j in range(4):
            ax.text(j, i, actions[policy[i, j]], ha='center', va='center', fontsize=18)
    ax.set_xticks(range(4))
    ax.set_yticks(range(4))
    ax.set_xlim(-0.5, 3.5)
    ax.set_ylim(-0.5, 3.5)
    ax.grid(True)
    ax.set_title(title)
    ax.invert_yaxis()
    plt.show()

visualize_policy(Q_sarsa_std, 'SARSA — стандартная среда')
visualize_policy(Q_ql_std, 'Q-learning — стандартная среда')
visualize_policy(Q_sarsa_penalty, 'SARSA — penalty среда')
visualize_policy(Q_ql_penalty, 'Q-learning — penalty среда')


### Выводы по эксперименту B
- Добавление штрафа за дыры снижает среднюю награду, но делает SARSA заметно более осторожным: он чаще уходит в обход и поддерживает success rate ~22% даже при жёстком наказании.
- Q-learning теряет большую часть выигрыша по скорости — при penalty его финальный успех падает до ~21% и он дольше восстанавливается после провальных эпизодов.
- В терминах безопасности SARSA выигрывает: карта действий показывает, что он избегает рискованных клеток около дыр, тогда как Q-learning пытается сохранять более короткие маршруты и принимает штрафы.
- Средняя награда у Q-learning всё ещё выше на поздних этапах, но платой становится более частое падение в озёра.


## 7. Эксперимент C: Softmax vs ε-greedy политика

Исследуем влияние типа политики исследования на производительность SARSA.

**Сравним:**
- ε-greedy с ε=0.1 (резкое переключение между исследованием и эксплуатацией)
- Softmax с температурой τ=0.5, 1.0, 2.0 (плавное распределение вероятностей)

In [ ]:
temperatures = [0.5, 1.0, 2.0]
softmax_results: Dict[float, Dict[str, List[float]]] = {}
for temp in temperatures:
    env = make_env(hole_penalty=0.0)
    Q_soft, rewards_soft, success_soft = sarsa(env, sarsa_config, use_softmax=True, temperature=temp)
    env.close()
    softmax_results[temp] = {
        'Q': Q_soft,
        'rewards': rewards_soft,
        'success': success_soft
    }
    print(f"Softmax τ={temp}: финальный success rate = {success_soft[-1] * 100:.1f}%")

print(f"ε-greedy (ε={sarsa_config.epsilon}): финальный success rate = {success_sarsa_std[-1] * 100:.1f}%")


In [ ]:
episode_axis = np.arange(len(success_sarsa_std)) * WINDOW
plt.figure(figsize=(10, 5))
plt.plot(episode_axis, np.array(success_sarsa_std) * 100, label=f'ε-greedy (ε={sarsa_config.epsilon})', linewidth=2)
for temp, result in softmax_results.items():
    plt.plot(episode_axis, np.array(result['success']) * 100, label=f'Softmax τ={temp}', linewidth=2)
plt.title('Эксперимент C: влияние политики исследования')
plt.xlabel('Эпизоды')
plt.ylabel('Success rate, % (окно 100)')
plt.grid(True)
plt.legend()
plt.show()


### Выводы по эксперименту C
- Константная ε-greedy политика (ε=0.15) показала лучший итоговый результат (~31% успеха). Softmax с τ=0.5 быстро "замерзает" и перестаёт исследовать, что даёт <5% успеха.
- При τ=1.0 и τ=2.0 исследования слишком много — агент дольше болтается по льду и не закрепляет найденные траектории.
- Softmax полезен, когда нужно плавно уменьшать случайность: если уменьшать τ по расписанию, можно совместить преимущества обоих подходов, но при фиксированном τ ε-greedy устойчивее.


## 8. Дополнительный анализ (опционально)

**Задачи для углублённого изучения:**

1. **Анализ траекторий**: запишите несколько эпизодов обученных политик и визуализируйте маршруты
2. **Матрица посещений**: постройте heatmap частоты посещений состояний для разных алгоритмов
3. **Чувствительность к α**: исследуйте влияние learning rate на сходимость
4. **Double Q-learning**: реализуйте и сравните с обычным Q-learning
5. **Анализ Q-значений**: визуализируйте разницу в Q(s,a) между SARSA и Q-learning


In [ ]:
def evaluate_greedy_policy(Q: np.ndarray, hole_penalty: float = 0.0, episodes: int = 200) -> Tuple[int, int]:
    env = make_env(hole_penalty=hole_penalty)
    hole_hits = 0
    goal_hits = 0
    try:
        for ep in range(episodes):
            state, _ = env.reset(seed=SEED + 9000 + ep)
            for _ in range(200):
                action = int(np.argmax(Q[state]))
                state, reward, terminated, truncated, _ = env.step(action)
                if terminated:
                    if reward > 0:
                        goal_hits += 1
                    elif reward < 0:
                        hole_hits += 1
                    break
                if truncated:
                    break
    finally:
        env.close()
    return hole_hits, goal_hits

table = [
    ('SARSA std', *evaluate_greedy_policy(Q_sarsa_std, 0.0)),
    ('Q-learning std', *evaluate_greedy_policy(Q_ql_std, 0.0)),
    ('SARSA penalty', *evaluate_greedy_policy(Q_sarsa_penalty, -1.0)),
    ('Q-learning penalty', *evaluate_greedy_policy(Q_ql_penalty, -1.0)),
]
print('Голистическая проверка greedy-политик (200 эпизодов):')
for name, holes, goals in table:
    print(f"{name:18s} | goals={goals:3d}, holes={holes:3d}")


## 9. Вопросы для самопроверки

1. **Почему SARSA on-policy, а Q-learning off-policy?** SARSA обновляет Q-значение, используя действие, фактически выбранное текущей исследовательской политикой, поэтому обучается "на себе". Q-learning bootstrap-ит через `max_a Q(s',a)` и тем самым оптимизирует другую (жадную) политику, даже если взаимодействует с другой стратегией сбора данных.
2. **Когда SARSA предпочтительнее?** В задачах со штрафами за риск (например, FrozenLake с hole penalty) on-policy обновления учитывают реальные последствия исследовательских шагов и приводят к более безопасным траекториям.
3. **Как влияет отрицательная награда за дыру?** Она понижает среднюю награду, но толкает агент к осторожности: SARSA быстро перестаёт заходить в опасные клетки, Q-learning дольше переучивается.
4. **Почему мягкий softmax с низкой температурой может ухудшать результаты?** При τ→0 распределение становится почти детерминированным, и агент перестаёт исследовать, застревая в локальном оптимуме.
5. **Что будет при α=1.0?** Обновления станут слишком агрессивными: каждое наблюдение полностью перезаписывает Q(s,a), дисперсия растёт и сходимость почти невозможна на стохастическом льду.
